# IAN 603 · Module 3
# PostgreSQL in Action

Read and change relational data safely with practical PostgreSQL queries.

Course module:  
https://ali2449.github.io/IAN-603-Preparing-Data-for-Analytics-/s_module3.html


## Learning goals

Work through the examples in order. Focus on the meaning of each row, the reason for each SQL clause, and how you would validate the result.


## 0. Install required Python packages

Run this only if the packages are not already installed.


In [ ]:
%pip install -q pandas sqlalchemy psycopg2-binary ipywidgets


## 1. Load the course database helper

This notebook uses a small course helper to open the PostgreSQL connection window and display query results.

**You only need this notebook file.** When you run the next cell, the helper is loaded automatically from the course GitHub repository if it is not already available.

If you would like your own copy of the helper, you can download it here:

[Download `ian603_database.py`](https://github.com/ali2449/IAN-603-Preparing-Data-for-Analytics-/raw/refs/heads/MSIA/ian603_database.py)

After the helper loads, continue to the connection section below.


In [ ]:
from pathlib import Path
import urllib.request

MODULE_FILE = Path("ian603_database.py")
MODULE_URL = (
    "https://raw.githubusercontent.com/"
    "ali2449/IAN-603-Preparing-Data-for-Analytics-/MSIA/"
    "ian603_database.py"
)

if not MODULE_FILE.exists():
    try:
        urllib.request.urlretrieve(MODULE_URL, MODULE_FILE)
        print("Course database helper downloaded.")
    except Exception as e:
        print("The database helper could not be downloaded automatically.")
        print("Use the download link in the Markdown cell above, then run this cell again.")
        raise

from ian603_database import IAN603Database

db = IAN603Database()

display_query = db.display_query
execute_query = db.execute_query
execute_script = db.execute_script

print("Database helper ready.")


## 2. Connect to PostgreSQL

Run the next cell to open the connection window. It works much like the connection settings you use in pgAdmin.

The usual local PostgreSQL defaults are already filled in:

| Setting | Default |
|---|---|
| Host | `localhost` |
| Database | `postgres` |
| Username | `postgres` |
| Port | `5432` |

Enter your PostgreSQL password, then click **Connect**.

If you are using your own computer, you may also choose to save the connection settings for the next time you open the notebook.


In [ ]:
db.show_connection_panel()


## 3. Check the connection

After you click **Connect**, run the next cell to make sure the notebook is ready to use PostgreSQL.


In [ ]:
if db.connected:
    db.test_connection()
else:
    print("No active PostgreSQL connection.")
    print("Return to the connection panel above and click Connect.")


## How to think about Module 3

Most SQL work follows a small workflow:

```text
question
  ↓
SELECT the rows
  ↓
filter / sort / calculate
  ↓
inspect the result
  ↓
only then modify data when needed
```

### Safe write habit

Before `UPDATE` or `DELETE`:

1. preview the target rows with `SELECT`,
2. verify the filter,
3. use `RETURNING` when useful,
4. use a transaction while testing.

This module also introduces JSONB and table-building operations, but the main goal is confidence with everyday PostgreSQL.


# Teaching database

Run the next cell to create or reset the Module 3 teaching data. You can return to this cell whenever you want a clean starting point.


In [ ]:
setup_sql = "-- ============================================================\n-- IAN 603 - MODULE 3\n-- PostgreSQL in Action\n-- ============================================================\n-- Companion file for s_module3.html\n-- PostgreSQL / pgAdmin Query Tool\n--\n-- Topics:\n-- SELECT, filtering, aliases, INSERT, UPDATE, DELETE, RETURNING,\n-- transactions, JSONB, CREATE TABLE AS, and MERGE.\n--\n-- SAFETY HABIT\n-- Before UPDATE or DELETE, preview the target rows with SELECT.\n-- Use RETURNING when useful, and use transactions when testing.\n-- ============================================================\n\n-- ============================================================\n-- SECTION 1 - CREATE / RESET TEACHING DATABASE\n-- ============================================================\n\nDROP TABLE IF EXISTS clinical_notes, admissions, doctors, patients CASCADE;\n    CREATE TABLE patients (\n      patient_id integer GENERATED ALWAYS AS IDENTITY PRIMARY KEY,\n      mrn text NOT NULL UNIQUE,\n      full_name text NOT NULL,\n      age smallint NOT NULL CHECK (age BETWEEN 0 AND 120),\n      gender text NOT NULL,\n      is_active boolean NOT NULL DEFAULT TRUE\n    );\n    CREATE TABLE doctors (\n      doctor_id integer GENERATED ALWAYS AS IDENTITY PRIMARY KEY,\n      full_name text NOT NULL,\n      specialty text NOT NULL\n    );\n    CREATE TABLE admissions (\n      admission_id integer GENERATED ALWAYS AS IDENTITY PRIMARY KEY,\n      patient_id integer NOT NULL REFERENCES patients(patient_id) ON DELETE RESTRICT,\n      doctor_id integer NOT NULL REFERENCES doctors(doctor_id) ON DELETE RESTRICT,\n      diagnosis text NOT NULL,\n      admission_date date NOT NULL,\n      discharge_date date,\n      status text NOT NULL DEFAULT 'Admitted',\n      CHECK (discharge_date IS NULL OR discharge_date >= admission_date)\n    );\n    CREATE TABLE clinical_notes (\n      note_id integer GENERATED ALWAYS AS IDENTITY PRIMARY KEY,\n      patient_id integer NOT NULL REFERENCES patients(patient_id) ON DELETE CASCADE,\n      note jsonb NOT NULL\n    );\n    INSERT INTO patients (mrn,full_name,age,gender,is_active) VALUES\n      ('MRN1001','Carol Nguyen',42,'Female',TRUE),\n      ('MRN1002','Bob Smith',68,'Male',TRUE),\n      ('MRN1003','Elena Garcia',34,'Female',TRUE),\n      ('MRN1004','David Lee',72,'Male',TRUE),\n      ('MRN1005','Amina Yusuf',58,'Female',TRUE),\n      ('MRN1006','Marcus Reed',27,'Male',TRUE);\n    INSERT INTO doctors (full_name,specialty) VALUES\n      ('Jordan Chen','Family Medicine'),\n      ('Priya Shah','Internal Medicine'),\n      ('Noah Bennett','Cardiology');\n    INSERT INTO admissions (patient_id,doctor_id,diagnosis,admission_date,discharge_date,status) VALUES\n      (1,1,'Asthma','2023-02-18','2023-02-20','Discharged'),\n      (2,3,'Chest pain','2023-03-04','2023-03-07','Discharged'),\n      (4,2,'Dehydration','2023-03-10',NULL,'Admitted'),\n      (5,1,'Migraine','2023-04-02','2023-04-03','Discharged'),\n      (3,2,'Pneumonia','2023-05-11',NULL,'Admitted');\n    INSERT INTO clinical_notes (patient_id,note) VALUES\n      (1,'{\"symptom\":\"wheezing\",\"severity\":\"moderate\",\"follow_up\":true}'::jsonb),\n      (2,'{\"symptom\":\"chest pain\",\"severity\":\"severe\",\"follow_up\":true}'::jsonb),\n      (3,'{\"symptom\":\"cough\",\"severity\":\"moderate\",\"follow_up\":true}'::jsonb),\n      (5,'{\"symptom\":\"headache\",\"severity\":\"mild\",\"follow_up\":false}'::jsonb);"

execute_script(setup_sql)


# QUICK TABLE CHECK


## QUICK TABLE CHECK · Step 1

Run this step and connect the result back to the module concept.


In [ ]:
query = "SELECT * FROM patients ORDER BY patient_id;"

display_query(query, title="QUICK TABLE CHECK · Step 1")


## QUICK TABLE CHECK · Step 2

Run this step and connect the result back to the module concept.


In [ ]:
query = "SELECT * FROM doctors ORDER BY doctor_id;"

display_query(query, title="QUICK TABLE CHECK · Step 2")


## QUICK TABLE CHECK · Step 3

Run this step and connect the result back to the module concept.


In [ ]:
query = "SELECT * FROM admissions ORDER BY admission_id;"

display_query(query, title="QUICK TABLE CHECK · Step 3")


## QUICK TABLE CHECK · Step 4

Run this step and connect the result back to the module concept.


In [ ]:
query = "SELECT * FROM clinical_notes ORDER BY note_id;"

display_query(query, title="QUICK TABLE CHECK · Step 4")


# RUNNABLE MODULE EXAMPLES


## Example 01

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "SELECT patient_id, mrn, full_name, age, gender, is_active\nFROM patients\nORDER BY patient_id;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Example 01")


## Example 02

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "SELECT mrn, full_name, age\nFROM patients\nORDER BY age DESC;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Example 02")


## Example 03

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "SELECT mrn,\n       full_name AS patient,\n       age,\n       age >= 65 AS is_senior\nFROM patients\nORDER BY age DESC;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Example 03")


## Example 04

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "SELECT mrn, full_name, age, is_active\nFROM patients\nWHERE is_active = TRUE\n  AND age >= 50\nORDER BY age DESC;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Example 04")


## Example 05

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "SELECT mrn, full_name, age, gender\nFROM patients\nWHERE gender = 'Female'\n  AND age BETWEEN 30 AND 60\nORDER BY age;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Example 05")


## Example 06

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "SELECT mrn, full_name\nFROM patients\nWHERE full_name ILIKE 'c%';\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Example 06")


## Example 07

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "SELECT admission_id, patient_id, diagnosis, admission_date, discharge_date\nFROM admissions\nWHERE discharge_date IS NULL\nORDER BY admission_date;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Example 07")


## Example 08

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "SELECT admission_id, patient_id, diagnosis, admission_date\nFROM admissions\nORDER BY admission_date DESC, admission_id DESC\nLIMIT 3;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Example 08")


## Example 09

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "INSERT INTO patients (mrn, full_name, age, gender)\nVALUES ('MRN1010', 'Ivy Carter', 50, 'Female')\nRETURNING patient_id, mrn, full_name, age, gender;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Example 09")


## Example 10

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "SELECT patient_id, mrn, full_name, age\nFROM patients\nWHERE mrn = 'MRN1003';\n\nUPDATE patients\nSET age = 35\nWHERE mrn = 'MRN1003'\nRETURNING patient_id, mrn, full_name, age;\n\n\n-- ------------------------------------------------------------"

execute_script(query)


## Example 11

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "UPDATE patients\nSET is_active = FALSE\nWHERE mrn = 'MRN1004'\nRETURNING patient_id, mrn, full_name, is_active;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Example 11")


## Example 12

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "DELETE FROM patients\nWHERE mrn = 'MRN1004';\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Example 12")


## Example 13

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "BEGIN;\n\nUPDATE patients\nSET age = 35\nWHERE mrn = 'MRN1003'\nRETURNING patient_id, mrn, full_name, age;\n\nROLLBACK;\n\nSELECT mrn, full_name, age\nFROM patients\nWHERE mrn = 'MRN1003';\n\n\n-- ------------------------------------------------------------"

execute_script(query)


## Example 14

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "-- 1. Start a transaction\n\n-- 2. Update MRN1005 to age 59 and use RETURNING\n\n-- 3. Roll back the transaction\n\n-- 4. SELECT MRN1005 again to verify the original age;\n\n\n-- ------------------------------------------------------------"

execute_script(query)


## Example 15

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "SELECT patient_id,\n       note->>'symptom' AS symptom,\n       note->>'severity' AS severity,\n       (note->>'follow_up')::boolean AS follow_up\nFROM clinical_notes\nWHERE note->>'severity' IN ('moderate', 'severe')\nORDER BY patient_id;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Example 15")


## Example 16

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "CREATE TABLE senior_patients (\n  patient_id integer PRIMARY KEY,\n  mrn text NOT NULL,\n  full_name text NOT NULL,\n  age smallint NOT NULL\n);\n\nINSERT INTO senior_patients (patient_id, mrn, full_name, age)\nSELECT patient_id, mrn, full_name, age\nFROM patients\nWHERE age >= 65;\n\nSELECT *\nFROM senior_patients\nORDER BY age DESC;\n\n\n-- ------------------------------------------------------------"

execute_script(query)


## Example 17

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "CREATE TEMP TABLE incoming_patients (\n  mrn text,\n  full_name text,\n  age smallint,\n  gender text\n);\n\nINSERT INTO incoming_patients VALUES\n('MRN1002', 'Bob Smith', 69, 'Male'),\n('MRN1007', 'Grace Hall', 55, 'Female');\n\nMERGE INTO patients AS target\nUSING incoming_patients AS source\nON target.mrn = source.mrn\nWHEN MATCHED THEN\n  UPDATE SET full_name = source.full_name,\n             age = source.age,\n             gender = source.gender\nWHEN NOT MATCHED THEN\n  INSERT (mrn, full_name, age, gender)\n  VALUES (source.mrn, source.full_name, source.age, source.gender);\n\nSELECT mrn, full_name, age, gender\nFROM patients\nWHERE mrn IN ('MRN1002','MRN1007')\nORDER BY mrn;\n\n\n-- ------------------------------------------------------------"

execute_script(query)


## Example 18

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "-- Choose Task A or Task B and write your SQL here.;\n\n\n-- ------------------------------------------------------------"

execute_script(query)


## Example 19

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "SELECT mrn, full_name, age\nFROM patients\nWHERE\nORDER BY age DESC;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Example 19")


## Example 20

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "INSERT INTO patients (mrn, full_name, age, gender)\nVALUES\nRETURNING mrn, full_name, age, gender;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Example 20")


## Example 21

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "UPDATE patients\nSET\nWHERE\nRETURNING mrn, full_name, age;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Example 21")


## Example 22

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "BEGIN;\n\n-- UPDATE MRN1005 here\n\n-- ROLLBACK here\n\nSELECT mrn, full_name, age\nFROM patients\nWHERE mrn = 'MRN1005';\n\n\n-- ------------------------------------------------------------"

execute_script(query)


## Example 23

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "SELECT patient_id,\n       note->>'symptom' AS symptom,\n       note->>'severity' AS severity\nFROM clinical_notes\nWHERE\nORDER BY patient_id;\n\n\n-- ------------------------------------------------------------"

display_query(query, title="Example 23")


## Example 24

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "${escapeText(saved)};"

execute_script(query)


# SAFE WRITE PATTERN


## SAFE WRITE PATTERN · Step 1

Run this step and connect the result back to the module concept.


In [ ]:
query = "-- Preview.\nSELECT patient_id, mrn, full_name, age\nFROM patients\nWHERE mrn = 'MRN1003';"

display_query(query, title="SAFE WRITE PATTERN · Step 1")


## SAFE WRITE PATTERN · Step 2

Run this step and connect the result back to the module concept.


In [ ]:
query = "-- Test inside a transaction.\nBEGIN;"

execute_script(query)


## SAFE WRITE PATTERN · Step 3

Run this step and connect the result back to the module concept.


In [ ]:
query = "UPDATE patients\nSET age = 35\nWHERE mrn = 'MRN1003'\nRETURNING patient_id, mrn, full_name, age;"

display_query(query, title="SAFE WRITE PATTERN · Step 3")


## SAFE WRITE PATTERN · Step 4

Run this step and connect the result back to the module concept.


In [ ]:
query = "ROLLBACK;"

execute_script(query)


## SAFE WRITE PATTERN · Step 5

Run this step and connect the result back to the module concept.


In [ ]:
query = "-- Verify original state.\nSELECT patient_id, mrn, full_name, age\nFROM patients\nWHERE mrn = 'MRN1003';"

display_query(query, title="SAFE WRITE PATTERN · Step 5")


# Practice

1. Return active patients age 50 or older.
2. Find patients whose name begins with C.
3. Insert a new patient and use RETURNING.
4. Test an UPDATE inside BEGIN / ROLLBACK.
5. Extract symptom and severity from the JSONB note.
6. Explain when MERGE is useful.
CHECKLIST
[ ] I know whether my statement reads, changes, or defines data.
[ ] I preview rows before UPDATE or DELETE.
[ ] I use stable identifiers in write conditions.
[ ] I understand COMMIT versus ROLLBACK.
[ ] I can query simple JSONB fields.

Try each task before looking back at the earlier examples.


In [ ]:
# Write your practice SQL here
practice_query = """

"""

# When ready:
# display_query(practice_query, title="Practice")


# Module 3 summary

Before moving on, make sure you can explain the main ideas in your own words and can describe the **grain** or row meaning of the important query results.

A good final check is:

1. What question did the query answer?
2. What did one input row represent?
3. What does one output row represent?
4. How did SQL change the data shape or meaning?
5. How would you validate that the result is correct?

## Next
Continue to **Module 4**.
